# Сагитталь — бинарная классификация (5-fold CV)

Логика обучения и метрик в репозитории: `MLService/training/sagittal_binary_cv.py`, датасеты и K-fold — в `training/datasets/tmj_position_dataset.py`, `training/tmj_position_label_table.py`, метрики — `training/utils/binary_metrics.py`, аугментации 3D — `training/utils/volume_aug_3d.py`.

**Yandex DataSphere:** датасет с кропами и метками с GitHub — см. [init_datasphere_dataset.ipynb](init_datasphere_dataset.ipynb). Пути по умолчанию задаёт `training/utils/datasphere_env.py` (`/home/jupyter/datasets/tmj/`, `detector_crops_v2`, JSON отчёта в `filestore/experiments/`). Переопределение: переменные среды `TMJ_DATASET_DIR`, `ML_SERVICE_ROOT`.

План экспериментов: `MLService/docs/superpowers/prompts/improve-sag-classifier-metrics.md`.

**Сплиты:** только train / validation внутри каждого фолда CV, отдельной тестовой выборки нет.

**Дефолты конфига:** backbone `[8,16,32,64]`, `fc_hidden=128`, train-аугментации `strong` (флипы + малые повороты + джиттер яркости).

Здесь: зависимости, `PYTHONPATH`, авто-пути (DataSphere / локально) и запуск `run_sagittal_binary_cv`.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "scikit-learn", "nibabel", "tqdm", "scipy"])

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_boot = None
for p in [_HERE, *list(_HERE.parents)[:10], Path("/home/jupyter/project/MasterProject/MLService"), Path("/content/MasterProject/MLService")]:
    if (p / "training" / "sagittal_binary_cv.py").is_file():
        _boot = p.resolve()
        break
if _boot is None:
    _boot = Path("/home/jupyter/project/MasterProject/MLService")
sys.path.insert(0, str(_boot))

from training.utils.datasphere_env import (
    default_cv_output_json,
    infer_mlservice_root,
    is_datasphere,
    sagittal_binary_cv_path_kwargs,
)

MLSERVICE_ROOT = infer_mlservice_root(_HERE)

print("MLService:", MLSERVICE_ROOT)
print("DataSphere:", is_datasphere(), "| crops:", sagittal_binary_cv_path_kwargs()["crop_dir"])

In [ ]:
from training.sagittal_binary_cv import SagittalBinaryCVConfig, run_sagittal_binary_cv, _print_cv_table

path_kw = sagittal_binary_cv_path_kwargs()
out_json = default_cv_output_json(MLSERVICE_ROOT)

cfg = SagittalBinaryCVConfig(
    **path_kw,
    epochs=80,
    batch_size=16,
    train_augment_mode="strong",
    features=(8, 16, 32, 64),
    fc_hidden=128,
    output_json=str(out_json),
    tqdm_disable=False,
)
print("Paths:", path_kw)
print("Output:", out_json)

result = run_sagittal_binary_cv(cfg)
_print_cv_table(result)